In [1]:
!pip install torch==2.9.0 torchvision==0.24.0 --index-url https://download.pytorch.org/whl/cu126

Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 137.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 228.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 184.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 85.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 101.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 161.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 120.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 105.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.7/124.7 MB 124.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 116.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 161.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1

In [2]:
## Load necessory Libraries

## ML Libraries
import numpy as np
import pandas as pd

## PyTorch libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

## Torch Vission Library
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

# Image processing libraries
from PIL import Image
import cv2

# Image augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

## Essentials
import os
from pathlib import Path

## Visualization Libraries
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

In [3]:
## Verify versions of libraries
print("📦 Library Versions:")
print(f"   • NumPy       : {np.__version__}")
print(f"   • Pandas      : {pd.__version__}")
print(f"   • PyTorch     : {torch.__version__}")
print(f"   • Torchvision : {torchvision.__version__}")
print(f"   • Matplotlib  : {plt.matplotlib.__version__}")
print(f"   • Seaborn     : {sns.__version__}")

print("—" * 60)

## Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpu = torch.cuda.device_count()
print(f"🚀 Using device: {device}")
print(f"   Number of GPU devices: {n_gpu}")

for i in range(n_gpu):
    props = torch.cuda.get_device_properties(i)
    total_mem_gb = props.total_memory / (1024 ** 3)
    print(f"\n   GPU {i}: {props.name}")
    print(f"        • Compute Capability : {props.major}.{props.minor}")
    print(f"        • Total Memory       : {total_mem_gb:.2f} GB")
    print(f"        • Multi Processors   : {props.multi_processor_count}")
    print(f"        • CUDA Cores (est.)  : depends on architecture")
    print("—" * 60)

📦 Library Versions:
   • NumPy       : 2.0.2
   • Pandas      : 2.3.3
   • PyTorch     : 2.9.0+cu126
   • Torchvision : 0.24.0+cu126
   • Matplotlib  : 3.10.0
   • Seaborn     : 0.13.2
————————————————————————————————————————————————————————————
🚀 Using device: cuda
   Number of GPU devices: 1

   GPU 0: Tesla P100-PCIE-16GB
        • Compute Capability : 6.0
        • Total Memory       : 15.89 GB
        • Multi Processors   : 56
        • CUDA Cores (est.)  : depends on architecture
————————————————————————————————————————————————————————————


In [4]:
## Environment setup and variable initialization
ROOT = Path.cwd() / "/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1"
IMAGE_DIR = ROOT / "images"

TRAIN_CSV = ROOT / "train.csv"
TEST_CSV = ROOT / "test.csv"
SUBMISSION_CSV = ROOT / "submission.csv"
SAMPLE_SUBMISSION_CSV = ROOT / "sample_submission.csv"

print(f"📁 Data Paths:")
print(f"   • Root Directory        : '{ROOT}'")
print(f"   • Image Directory       : '{IMAGE_DIR}'")
print(f"   • Training CSV          : '{TRAIN_CSV}'")
print(f"   • Test CSV              : '{TEST_CSV}'")
print(f"   • Submission CSV        : '{SUBMISSION_CSV}'")
print(f"   • Sample Submission CSV : '{SAMPLE_SUBMISSION_CSV}'")

# Set random seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

📁 Data Paths:
   • Root Directory        : '/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1'
   • Image Directory       : '/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/images'
   • Training CSV          : '/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/train.csv'
   • Test CSV              : '/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/test.csv'
   • Submission CSV        : '/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/submission.csv'
   • Sample Submission CSV : '/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/sample_submission.csv'


In [5]:
## Data Inspection
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print("📊 Training Data Overview:")
print(f"   • Number of samples : {len(train_df)}")
print(f"   • Number of Columns : {len(train_df.columns)}")

print("\n📊 Test Data Overview:")
print(f"   • Number of samples : {len(test_df)}")
print(f"   • Number of Columns : {len(test_df.columns)}")

CLASSES = train_df.columns[1:].tolist()
print(f"\n🎯 Target Classes:")
for idx, cls in enumerate(CLASSES):
    print(f"   {(idx+1):>3}. {cls}")

📊 Training Data Overview:
   • Number of samples : 51043
   • Number of Columns : 21

📊 Test Data Overview:
   • Number of samples : 17015
   • Number of Columns : 1

🎯 Target Classes:
     1. Atelectasis
     2. Cardiomegaly
     3. Consolidation
     4. Edema
     5. Effusion
     6. Emphysema
     7. Fibrosis
     8. Hernia
     9. Infiltration
    10. Mass
    11. Nodule
    12. Pleural_Thickening
    13. Pneumonia
    14. Pneumothorax
    15. Pneumoperitoneum
    16. Pneumomediastinum
    17. Subcutaneous Emphysema
    18. Tortuous Aorta
    19. Calcification of the Aorta
    20. No Finding


In [6]:
## Utility Function for class-to-index and index-to-class mappings
CLASSES = train_df.columns[1:]
num_classes  = len(CLASSES)

class_to_idx = {cls: i for i, cls in enumerate(CLASSES)}
idx_to_class = {i: cls for cls, i in class_to_idx.items()}
no_finding_idx = list(CLASSES).index("No Finding") if "No Finding" in list(CLASSES) else None

def onehot_to_label(onehot_vector):
    return int(np.argmax(onehot_vector))

def label_to_onehot(y, num_classes):
    return np.eye(num_classes)[[y]][0]

def labels_to_onehot(y, num_classes):
    return np.array([label_to_onehot(label, num_classes) for label in y])

def df_onehot_to_label(df):
    df = df.copy()
    df["label"] = df[CLASSES].values.argmax(axis=1)
    df = df.drop(columns=CLASSES)
    return df

def df_label_to_onehot(df):
    df = df.copy()
    
    onehot = np.zeros((len(df), len(CLASSES)), dtype=int)
    onehot[np.arange(len(df)), df["label"].values] = 1
    onehot_df = pd.DataFrame(onehot, columns=CLASSES, index=df.index)
    
    df = df.drop(columns=["label"])
    df = pd.concat([df, onehot_df], axis=1)
    return df

## Data Preperation

In [7]:
# Split into Train/Val
from sklearn.model_selection import train_test_split

X = train_df.drop(columns=CLASSES)
y = df_onehot_to_label(train_df)["label"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

In [8]:
# Dataset Class
class ChestXrayDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, no_label=False):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform
        self.no_label = no_label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]["id"]
        img_path = self.image_dir / img_name
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.no_label:
            return image
        else:
            label = self.df.iloc[idx]['label'] if 'label' in self.df.columns else onehot_to_label(
                self.df.iloc[idx][CLASSES].values)

            return image, label

In [9]:
## Transformations Based on ResNet50
IMAGE_SIZE = 256                 # Original size
CROP_SIZE = 224
MEAN = [0.485, 0.456, 0.406]     # Calculated mean
STD = [0.229, 0.224, 0.225]      # Calculated std

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.CenterCrop((CROP_SIZE, CROP_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.CenterCrop((CROP_SIZE, CROP_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

In [10]:
## Create PyTorch Datasets Objects
train_dataset = ChestXrayDataset(
    df=pd.concat([X_train, pd.DataFrame({"label": y_train})], axis=1),
    image_dir=IMAGE_DIR,
    transform=train_transform
)

val_dataset = ChestXrayDataset(
    df=pd.concat([X_val, pd.DataFrame({"label": y_val})], axis=1),
    image_dir=IMAGE_DIR,
    transform=val_transform
)

test_dataset = ChestXrayDataset(
    df=test_df,
    image_dir=IMAGE_DIR,
    transform=val_transform,
    no_label=True
)

## Verify outputs
print("Shape of train dataset sample:", train_dataset[0][0].shape, "x", len(train_dataset))
print("Shape of val dataset sample  :", val_dataset[0][0].shape, "x", len(val_dataset))
print("Shape of test dataset sample: ", test_dataset[0].shape, "x", len(test_dataset))

Shape of train dataset sample: torch.Size([3, 224, 224]) x 40834
Shape of val dataset sample  : torch.Size([3, 224, 224]) x 10209
Shape of test dataset sample:  torch.Size([3, 224, 224]) x 17015


In [11]:
## Weighted Sampler
class_counts = np.bincount(y_train)
class_weights = 1.0 / (class_counts + 1e-6)
class_weights = np.sqrt(class_weights)

sample_weights = class_weights[y_train]
sample_weights = torch.from_numpy(sample_weights).float()

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

In [12]:
## Create PyTorch DataLoaders
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    shuffle=False,
    num_workers=4
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4
)

## Metrices

In [13]:
from sklearn.metrics import f1_score, accuracy_score

## Prepare cost matrix
cost_matrix = torch.ones((num_classes, num_classes)) * 7.0
cost_matrix.fill_diagonal_(-1.0)
cost_matrix[no_finding_idx, :] = 2.0    # FP side
cost_matrix[:, no_finding_idx] = 6.0    # FN side
cost_matrix[no_finding_idx, no_finding_idx] = -1.0

def compute_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro")

def competition_score(y_true, y_pred, num_classes=num_classes):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_true = labels_to_onehot(y_true, num_classes) if y_true.ndim == 1 else y_true
    y_pred = labels_to_onehot(y_pred, num_classes) if y_pred.ndim == 1 else y_pred

    class_scores = []
    for c in range(num_classes):
        true_c = y_true[:, c]
        pred_c = y_pred[:, c]

        N_c = np.sum(true_c)
        if N_c == 0:
            class_scores.append(0.0)
            continue
        TP = np.sum((true_c == 1) & (pred_c == 1))
        FP = np.sum((true_c == 0) & (pred_c == 1))
        FN = np.sum((true_c == 1) & (pred_c == 0))
        class_scores.append((TP - FP - 5 * FN) / N_c)
    return np.mean(class_scores)

## Model Training - Transfer Learning

In [14]:
## ResNet101 - Complex; Last one (fc(2048))
model = models.resnet101(weights=models.ResNet101_Weights)

# Freeze all layers (Except layer4 - Kept it to better generalize the model)
for name, param in model.named_parameters():
    if "layer4" not in name:
        param.requires_grad = False

# Replace the classifier
num_features = 2048

model.fc = nn.Sequential(
    nn.Linear(num_features, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, num_classes),
)

## Send the model to device
model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet101-63fe2227.pth" to /root/.cache/torch/hub/checkpoints/resnet101-63fe2227.pth


100%|██████████| 171M/171M [00:00<00:00, 223MB/s]


In [15]:
## Loss Functions
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none')

        pt = torch.exp(-ce_loss)  # probability of correct class
        focal_loss = (1 - pt) ** self.gamma * ce_loss

        if self.alpha is not None:
            at = self.alpha[targets]
            focal_loss = at * focal_loss

        return focal_loss.mean()

class AsymmetricCostLoss(nn.Module):
    def __init__(self, num_classes, no_finding_idx, weight=None):
        super().__init__()
        self.num_classes = num_classes
        self.no_finding_idx = no_finding_idx
        self.register_buffer("weight", weight)

    def forward(self, logits, target):
        costs = cost_matrix.to(device)[target]
        probs = torch.softmax(logits, dim=1)
        expected_cost = (probs * costs).sum(dim=1)
        if self.weight is not None:
            expected_cost = expected_cost * self.weight[target]
        return expected_cost.mean()

class_counts = np.bincount(y_train)
weights = 1.0 / (class_counts + 1e-6)
weights = weights / weights.sum() * len(weights)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

# criterion = FocalLoss(alpha=class_weights, gamma=2.0)
# criterion = AsymmetricCostLoss(num_classes=num_classes, no_finding_idx=no_finding_idx, weight=class_weights).to(device)
# criterion = nn.CrossEntropyLoss(weight=class_weights)
criterion = nn.CrossEntropyLoss()

In [16]:
## Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    patience=2,
    factor=0.3
)

In [17]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0

    for images, labels in tqdm(loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)

In [18]:
def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            # preds = torch.argmax(outputs, dim=1)
            probs = torch.softmax(outputs, dim=1)  # [B, C]
            costs = cost_matrix.to(outputs.device)  # [C, C]
            expected_cost = torch.matmul(probs, costs.T)  # [B, C]
            alpha = 0.7
            scores = alpha * (-expected_cost) + (1 - alpha) * probs
            preds = torch.argmax(scores, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    f1 = compute_f1(all_labels, all_preds)
    class_score = competition_score(all_labels, all_preds)
    accuracy = accuracy_score(all_labels, all_preds)

    return total_loss / len(loader), f1, class_score, accuracy, all_preds, all_labels

In [19]:
EPOCHS = 10
best_f1 = 0
best_class_score = float('-inf')

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_f1, val_class_score, val_accuracy, all_preds, all_labels = validate(model, val_loader, criterion)

    scheduler.step(val_loss)

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f} | Val Score: {val_class_score:.4f} | Val Accuracy: {val_accuracy:.4f}")
    unique, counts = np.unique(all_preds, return_counts=True)
    print(dict(zip(unique.tolist(), counts.tolist())))
    
    if val_class_score > best_class_score:
        best_class_score = val_class_score
        torch.save(model.state_dict(), "best_model.pth")
        print("✅ Saved Best Model")


Epoch 1/10


  0%|          | 0/1277 [00:00<?, ?it/s]

Train Loss: 2.4396 | Val Loss: 1.5263 | Val F1: 0.0400 | Val Score: -4.7249 | Val Accuracy: 0.6676
{19: 10209}
✅ Saved Best Model

Epoch 2/10


  0%|          | 0/1277 [00:00<?, ?it/s]

Train Loss: 2.2732 | Val Loss: 1.5079 | Val F1: 0.0418 | Val Score: -4.7212 | Val Accuracy: 0.6674
{4: 21, 19: 10188}
✅ Saved Best Model

Epoch 3/10


...

  0%|          | 0/1277 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c6367784720>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1637, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c6367784720>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Train Loss: 1.8051 | Val Loss: 1.4476 | Val F1: 0.0867 | Val Score: -4.6439 | Val Accuracy: 0.6655
{0: 32, 1: 79, 3: 80, 4: 141, 5: 3, 6: 3, 8: 2, 9: 17, 10: 5, 13: 49, 14: 1, 16: 2, 17: 5, 19: 9790}
✅ Saved Best Model

Epoch 10/10


  0%|          | 0/1277 [00:00<?, ?it/s]

Train Loss: 1.7398 | Val Loss: 1.4483 | Val F1: 0.0836 | Val Score: -4.7040 | Val Accuracy: 0.6631
{0: 28, 1: 117, 3: 24, 4: 134, 5: 8, 6: 7, 7: 2, 8: 3, 9: 11, 10: 8, 13: 60, 14: 1, 16: 2, 17: 32, 19: 9772}


## Fine-Tune

In [20]:
# Unfreeze entire model
for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    patience=2,
    factor=0.3
)

print("\n🔁 Starting Fine-Tuning...\n")

for epoch in range(5):
    print(f"\nFine-tune Epoch {epoch+1}/5")

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_f1, val_class_score, val_accuracy, all_preds, all_labels = validate(model, val_loader, criterion)

    scheduler.step(val_loss)

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f} | Val Score: {val_class_score:.4f} | Val Accuracy: {val_accuracy:.4f}")
    unique, counts = np.unique(all_preds, return_counts=True)
    print(dict(zip(unique.tolist(), counts.tolist())))
    
    if val_class_score > best_class_score:
        best_class_score = val_class_score
        torch.save(model.state_dict(), "best_model.pth")
        print("✅ Saved Best Model (Fine-tuned)")


🔁 Starting Fine-Tuning...


Fine-tune Epoch 1/5


  0%|          | 0/1277 [00:00<?, ?it/s]

Train Loss: 1.6459 | Val Loss: 1.4273 | Val F1: 0.1008 | Val Score: -4.6210 | Val Accuracy: 0.6639
{0: 83, 1: 96, 3: 30, 4: 151, 5: 12, 6: 16, 7: 1, 8: 9, 9: 21, 10: 25, 13: 71, 14: 2, 16: 1, 17: 14, 19: 9677}
✅ Saved Best Model (Fine-tuned)

Fine-tune Epoch 2/5


  0%|          | 0/1277 [00:00<?, ?it/s]

Train Loss: 1.5165 | Val Loss: 1.4157 | Val F1: 0.1042 | Val Score: -4.6122 | Val Accuracy: 0.6618
{0: 151, 1: 81, 3: 63, 4: 159, 5: 11, 6: 16, 7: 1, 8: 9, 9: 44, 10: 9, 13: 74, 14: 2, 16: 1, 17: 10, 18: 1, 19: 9577}
✅ Saved Best Model (Fine-tuned)

Fine-tune Epoch 3/5


...

Train Loss: 1.3828 | Val Loss: 1.4643 | Val F1: 0.1095 | Val Score: -4.6718 | Val Accuracy: 0.6592
{0: 123, 1: 115, 2: 5, 3: 59, 4: 182, 5: 10, 6: 33, 7: 4, 8: 1, 9: 49, 10: 9, 11: 7, 12: 1, 13: 130, 14: 2, 16: 4, 17: 24, 18: 1, 19: 9450}

Fine-tune Epoch 4/5


  0%|          | 0/1277 [00:00<?, ?it/s]

Train Loss: 1.2559 | Val Loss: 1.4459 | Val F1: 0.1175 | Val Score: -4.5988 | Val Accuracy: 0.6611
{0: 125, 1: 78, 2: 23, 3: 54, 4: 242, 5: 18, 6: 34, 7: 2, 8: 24, 9: 68, 10: 29, 11: 5, 12: 2, 13: 132, 14: 1, 16: 1, 17: 5, 18: 4, 19: 9362}
✅ Saved Best Model (Fine-tuned)

Fine-tune Epoch 5/5


  0%|          | 0/1277 [00:00<?, ?it/s]

Train Loss: 1.1360 | Val Loss: 1.4855 | Val F1: 0.1152 | Val Score: -4.6423 | Val Accuracy: 0.6545
{0: 101, 1: 71, 2: 45, 3: 30, 4: 302, 5: 46, 6: 33, 8: 23, 9: 131, 10: 36, 11: 20, 12: 3, 13: 132, 16: 1, 17: 14, 18: 1, 19: 9220}


In [21]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

test_preds = []

with torch.no_grad():
    for images in test_loader:
        images = images.to(device)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        test_preds.extend(preds.cpu().numpy())

print("No. of test predection:", len(test_preds))

No. of test predection: 17015


In [22]:
submission = test_df
submission['label'] = test_preds

submission = df_label_to_onehot(submission)
submission.to_csv("submission.csv", index=False)
print("Submission DF:", submission.shape)
print("✅ Submission file saved!")

Submission DF: (17015, 21)
✅ Submission file saved!
